In [ ]:
pip install statsbombpy

In [ ]:
from statsbombpy import sb

matches = sb.matches(
    competition_id=43,
    season_id=106
)

events = sb.competition_events(
    country="International",
    division="FIFA World Cup",
    season="2022",
    gender="male"
)

/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [ ]:
print(events.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234637 entries, 0 to 234636
Columns: 114 entries, 50_50 to under_pressure
dtypes: float64(8), int64(8), object(98)
memory usage: 204.1+ MB
None


In [ ]:
events.columns.tolist()

['50_50',
 'bad_behaviour_card',
 'ball_receipt_outcome',
 'ball_recovery_offensive',
 'ball_recovery_recovery_failure',
 'block_deflection',
 'block_offensive',
 'block_save_block',
 'carry_end_location',
 'clearance_aerial_won',
 'clearance_body_part',
 'clearance_head',
 'clearance_left_foot',
 'clearance_other',
 'clearance_right_foot',
 'counterpress',
 'dribble_no_touch',
 'dribble_nutmeg',
 'dribble_outcome',
 'dribble_overrun',
 'duel_outcome',
 'duel_type',
 'duration',
 'foul_committed_advantage',
 'foul_committed_card',
 'foul_committed_offensive',
 'foul_committed_penalty',
 'foul_committed_type',
 'foul_won_advantage',
 'foul_won_defensive',
 'foul_won_penalty',
 'goalkeeper_body_part',
 'goalkeeper_end_location',
 'goalkeeper_lost_in_play',
 'goalkeeper_outcome',
 'goalkeeper_position',
 'goalkeeper_punched_out',
 'goalkeeper_shot_saved_off_target',
 'goalkeeper_shot_saved_to_post',
 'goalkeeper_success_in_play',
 'goalkeeper_technique',
 'goalkeeper_type',
 'half_start_l

In [ ]:
print(events["type"].value_counts())

type
Pass                 68515
Ball Receipt*        63699
Carry                53764
Pressure             16554
Ball Recovery         5821
Duel                  4389
Clearance             2684
Block                 2386
Dribble               1793
Goal Keeper           1790
Foul Committed        1775
Miscontrol            1755
Foul Won              1693
Shot                  1494
Dispossessed          1431
Interception          1371
Dribbled Past         1036
Substitution           587
Injury Stoppage        403
Half End               286
Half Start             286
Tactical Shift         243
50/50                  236
Referee Ball-Drop      162
Starting XI            128
Shield                 104
Player On               74
Player Off              74
Bad Behaviour           44
Error                   28
Offside                 26
Own Goal Against         3
Own Goal For             3
Name: count, dtype: int64


In [ ]:
events.loc[
    events["type"] == "Pass",
    "pass_outcome"
].value_counts(dropna=False)

,count
pass_outcome,
NaN,56346
Incomplete,10393
Out,1116
Unknown,394
Pass Offside,224
Injury Clearance,42


In [ ]:
events.loc[
    events["type"] == "Shot",
    "shot_outcome"
].value_counts(dropna=False)

,count
shot_outcome,
Off T,448
Blocked,379
Saved,346
Goal,195
Wayward,85
Post,36
Saved to Post,3
Saved Off Target,2


In [ ]:
events.loc[
    events["type"] == "Shot",
    "shot_statsbomb_xg"
].describe()

,shot_statsbomb_xg
count,1494.000000
mean,0.125851
std,0.184223
min,0.000180
25%,0.029568
50%,0.056184
75%,0.118181
max,0.897468


In [ ]:
events["player"].isna().sum()

np.int64(1108)

In [ ]:
player_events = events[
    (events["player_id"].notna()) &
    (events["period"] < 5)
].copy()

In [ ]:
player_match_stats = (
    player_events
    .groupby(
        [
            "match_id",
            "player_id",
            "player",
            "team_id",
            "team"
        ],
        dropna=False
    )
    .agg(
        passes=("type", lambda x: (x == "Pass").sum()),
        shots=("type", lambda x: (x == "Shot").sum()),
        carries=("type", lambda x: (x == "Carry").sum()),
        pressures=("type", lambda x: (x == "Pressure").sum()),
        ball_recoveries=("type", lambda x: (x == "Ball Recovery").sum()),
        duels=("type", lambda x: (x == "Duel").sum()),
        clearances=("type", lambda x: (x == "Clearance").sum()),
        blocks=("type", lambda x: (x == "Block").sum()),
        dribbles=("type", lambda x: (x == "Dribble").sum()),
        interceptions=("type", lambda x: (x == "Interception").sum()),
        fouls_committed=("type", lambda x: (x == "Foul Committed").sum()),
        fouls_won=("type", lambda x: (x == "Foul Won").sum()),
        dispossessed=("type", lambda x: (x == "Dispossessed").sum()),
        miscontrols=("type", lambda x: (x == "Miscontrol").sum()),
    )
    .reset_index()
)

In [ ]:
pass_events = player_events[
    player_events["type"] == "Pass"
].copy()

shot_events = player_events[
    player_events["type"] == "Shot"
].copy()

In [ ]:
pass_stats = (
    pass_events
    .groupby(["match_id", "player_id"])
    .agg(
        completed_passes=(
            "pass_outcome", lambda x: x.isna().sum()
        ),
        incomplete_passes=(
            "pass_outcome", lambda x: (x == "Incomplete").sum()
        ),
        pass_out=(
            "pass_outcome", lambda x: (x == "Out").sum()
        ),
    )
    .reset_index()
)

shot_stats = (
    shot_events
    .groupby(["match_id", "player_id"])
    .agg(
        goals=(
            "shot_outcome", lambda x: (x == "Goal").sum()
        ),
        shots_on_target=(
            "shot_outcome", lambda x: x.isin(["Goal", "Saved"]).sum()
        ),
        xG=(
            "shot_statsbomb_xg", "sum"
        ),
    )
    .reset_index()
)

In [ ]:
player_match_stats = player_match_stats.merge(
    pass_stats,
    on=["match_id", "player_id"],
    how="left"
)

player_match_stats = player_match_stats.merge(
    shot_stats,
    on=["match_id", "player_id"],
    how="left"
)

player_match_stats.fillna(0, inplace=True)

In [ ]:
player_match_stats.head()

,match_id,player_id,player,team_id,team,passes,shots,carries,pressures,ball_recoveries,...,fouls_committed,fouls_won,dispossessed,miscontrols,completed_passes,incomplete_passes,pass_out,goals,shots_on_target,xG
0,3857254,3043.0,Christian Dannemann Eriksen,776,Denmark,80,1,70,10,4,...,2,0,2,2,66.0,12.0,1.0,0.0,1.0,0.085096
1,3857254,3557.0,Naïm Sliti,777,Tunisia,10,0,14,3,3,...,0,1,0,2,6.0,2.0,2.0,0.0,0.0,0.000000
2,3857254,3570.0,Pierre-Emile Højbjerg,776,Denmark,74,1,70,9,5,...,0,1,3,1,68.0,4.0,2.0,0.0,1.0,0.008709
3,3857254,3767.0,Ellyes Joris Skhiri,777,Tunisia,35,2,26,21,9,...,1,1,0,0,30.0,5.0,0.0,0.0,0.0,0.100992
4,3857254,3815.0,Kasper Schmeichel,776,Denmark,29,0,16,0,7,...,0,0,0,0,22.0,5.0,1.0,0.0,0.0,0.000000


In [ ]:
player_match_stats.to_csv(
    "player_match_stats_v1.csv",
    index=False
)

In [ ]:
events[
    (events["type"] == "Shot") &
    (events["shot_outcome"] == "Goal") &
    (events["player"].isin([
        "Lionel Andrés Messi Cuccittini",
        "Kylian Mbappé Lottin"
    ]))
][
    [
        "match_id",
        "period",
        "minute",
        "second",
        "player",
        "shot_outcome"
    ]
].sort_values(
    ["player", "match_id", "period", "minute", "second"]
)

,match_id,period,minute,second,player,shot_outcome
212121,3857266,2,60,25,Kylian Mbappé Lottin,Goal
212131,3857266,2,85,9,Kylian Mbappé Lottin,Goal
213239,3857279,2,67,3,Kylian Mbappé Lottin,Goal
212945,3869152,2,73,56,Kylian Mbappé Lottin,Goal
212947,3869152,2,90,5,Kylian Mbappé Lottin,Goal
212384,3869685,2,79,24,Kylian Mbappé Lottin,Goal
212385,3869685,2,80,59,Kylian Mbappé Lottin,Goal
212399,3869685,4,117,5,Kylian Mbappé Lottin,Goal
212402,3869685,5,120,13,Kylian Mbappé Lottin,Goal
212679,3857289,2,63,32,Lionel Andrés Messi Cuccittini,Goal
